# Translate English QA/RAG triplets to Vietnamese

This CPU notebook translates every `anchor`, `positive`, and `hard_negative` from the private SQLite input dataset. It writes checkpoints to a new SQLite database in `/kaggle/working`, so reruns resume unfinished or failed rows.

> `googletrans` uses an unofficial Google Translate endpoint. Keep this notebook private, expect rate limits, and use the retry controls below.

In [ ]:
!pip -q install 'googletrans==4.0.2'

In [ ]:
from __future__ import annotations

import asyncio
import random
import sqlite3
import time
from datetime import datetime, timezone
from pathlib import Path

from googletrans import Translator
from tqdm.auto import tqdm

INPUT_CANDIDATES = list(Path('/kaggle/input').glob('*/english_rag_qa_triplet_clean.db'))
if len(INPUT_CANDIDATES) != 1:
    raise FileNotFoundError(f'Expected one input database, found: {INPUT_CANDIDATES}')

SOURCE_DB = INPUT_CANDIDATES[0]
OUTPUT_DB = Path('/kaggle/working/english_rag_qa_triplet_vi.db')
SOURCE_TABLE = 'english_rag_qa_triplet_clean'
TARGET_TABLE = 'english_rag_qa_triplet_vi'

# Lower this to 8 if Google begins returning 429 errors.
CONCURRENCY = 16
RECORD_BATCH_SIZE = 96
MAX_CHARS_PER_REQUEST = 4_300
MAX_RETRIES = 7
RETRY_BASE_SECONDS = 1.5
TRANSLATION_LIMIT = None  # Set e.g. 500 for a smoke test.

print(f'Input: {SOURCE_DB}')
print(f'Output: {OUTPUT_DB}')

In [ ]:
def split_text(text: str, max_chars: int = MAX_CHARS_PER_REQUEST) -> list[str]:
    """Split long passages at whitespace while preserving every character."""
    if len(text) <= max_chars:
        return [text]
    chunks: list[str] = []
    remaining = text
    while len(remaining) > max_chars:
        cut = remaining.rfind(' ', 0, max_chars + 1)
        if cut < max_chars // 2:
            cut = max_chars
        chunks.append(remaining[:cut])
        remaining = remaining[cut:]
    chunks.append(remaining)
    return chunks


async def translate_text(translator: Translator, semaphore: asyncio.Semaphore, text: str) -> str:
    """Translate one field, retrying transient Google endpoint failures."""
    translated_chunks: list[str] = []
    for chunk in split_text(text):
        for attempt in range(MAX_RETRIES):
            try:
                async with semaphore:
                    result = await translator.translate(chunk, src='en', dest='vi')
                translated_chunks.append(result.text)
                break
            except Exception as error:
                if attempt == MAX_RETRIES - 1:
                    raise RuntimeError(f'translation failed after {MAX_RETRIES} attempts: {error}') from error
                delay = RETRY_BASE_SECONDS * (2 ** attempt) + random.random()
                await asyncio.sleep(delay)
    return ''.join(translated_chunks)


async def translate_record(
    translator: Translator, semaphore: asyncio.Semaphore, row: sqlite3.Row
) -> tuple[dict[str, str], str | None]:
    """Translate all three fields of a retrieval triplet concurrently."""
    try:
        anchor, positive, hard_negative = await asyncio.gather(
            translate_text(translator, semaphore, row['anchor']),
            translate_text(translator, semaphore, row['positive']),
            translate_text(translator, semaphore, row['hard_negative']),
        )
        return {'anchor': anchor, 'positive': positive, 'hard_negative': hard_negative}, None
    except Exception as error:
        return {}, str(error)[:2_000]


In [ ]:
source_connection = sqlite3.connect(f'file:{SOURCE_DB}?mode=ro', uri=True)
source_connection.row_factory = sqlite3.Row
output_connection = sqlite3.connect(OUTPUT_DB)
output_connection.execute('PRAGMA journal_mode=WAL')
output_connection.execute('PRAGMA synchronous=NORMAL')
output_connection.execute(f'''
CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
    data_id TEXT PRIMARY KEY,
    source TEXT NOT NULL,
    source_revision TEXT NOT NULL,
    source_split TEXT NOT NULL,
    source_row_index INTEGER NOT NULL,
    domain TEXT NOT NULL,
    task_type TEXT NOT NULL,
    anchor_en TEXT NOT NULL,
    positive_en TEXT NOT NULL,
    hard_negative_en TEXT NOT NULL,
    anchor TEXT,
    positive TEXT,
    hard_negative TEXT,
    translation_status TEXT NOT NULL,
    error_message TEXT,
    translated_at TEXT
)
''')
output_connection.execute(f'CREATE INDEX IF NOT EXISTS idx_{TARGET_TABLE}_status ON {TARGET_TABLE}(translation_status)')
output_connection.commit()

source_count = source_connection.execute(f'SELECT COUNT(*) FROM {SOURCE_TABLE}').fetchone()[0]
done_count = output_connection.execute(
    f"SELECT COUNT(*) FROM {TARGET_TABLE} WHERE translation_status = 'done'"
).fetchone()[0]
print(f'Source rows: {source_count:,}; already translated: {done_count:,}')

In [ ]:
SELECT_PENDING = f'''
SELECT s.*
FROM {SOURCE_TABLE} AS s
LEFT JOIN {TARGET_TABLE} AS t ON t.data_id = s.data_id
WHERE t.data_id IS NULL OR t.translation_status != 'done'
ORDER BY s.source, s.source_row_index
'''

INSERT_RESULT = f'''
INSERT INTO {TARGET_TABLE} (
    data_id, source, source_revision, source_split, source_row_index, domain, task_type,
    anchor_en, positive_en, hard_negative_en, anchor, positive, hard_negative,
    translation_status, error_message, translated_at
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
ON CONFLICT(data_id) DO UPDATE SET
    anchor = excluded.anchor,
    positive = excluded.positive,
    hard_negative = excluded.hard_negative,
    translation_status = excluded.translation_status,
    error_message = excluded.error_message,
    translated_at = excluded.translated_at
'''


def write_results(rows: list[sqlite3.Row], results: list[tuple[dict[str, str], str | None]]) -> None:
    now = datetime.now(timezone.utc).isoformat()
    payload = []
    for row, (translated, error) in zip(rows, results, strict=True):
        payload.append((
            row['data_id'], row['source'], row['source_revision'], row['source_split'],
            row['source_row_index'], row['domain'], row['task_type'],
            row['anchor'], row['positive'], row['hard_negative'],
            translated.get('anchor'), translated.get('positive'), translated.get('hard_negative'),
            'done' if error is None else 'error', error, now,
        ))
    output_connection.executemany(INSERT_RESULT, payload)
    output_connection.commit()


async def run_translation() -> None:
    pending_total = source_connection.execute(
        f'''SELECT COUNT(*) FROM {SOURCE_TABLE} AS s LEFT JOIN {TARGET_TABLE} AS t
        ON t.data_id = s.data_id WHERE t.data_id IS NULL OR t.translation_status != 'done' '''
    ).fetchone()[0]
    if TRANSLATION_LIMIT is not None:
        pending_total = min(pending_total, TRANSLATION_LIMIT)
    semaphore = asyncio.Semaphore(CONCURRENCY)
    pending_cursor = source_connection.execute(SELECT_PENDING)
    remaining = pending_total
    print(f'Pending rows this run: {pending_total:,}')
    async with Translator(service_urls=['translate.googleapis.com']) as translator:
        with tqdm(total=pending_total) as progress:
            while remaining > 0:
                rows = pending_cursor.fetchmany(min(RECORD_BATCH_SIZE, remaining))
                if not rows:
                    break
                results = await asyncio.gather(*(
                    translate_record(translator, semaphore, row) for row in rows
                ))
                write_results(rows, results)
                remaining -= len(rows)
                progress.update(len(rows))


started_at = time.monotonic()
await run_translation()
elapsed_minutes = (time.monotonic() - started_at) / 60
summary = output_connection.execute(
    f'SELECT translation_status, COUNT(*) FROM {TARGET_TABLE} GROUP BY translation_status'
).fetchall()
print(f'Elapsed: {elapsed_minutes:.1f} minutes; status: {summary}')

In [ ]:
done_count = output_connection.execute(
    f"SELECT COUNT(*) FROM {TARGET_TABLE} WHERE translation_status = 'done'"
).fetchone()[0]
invalid_count = output_connection.execute(
    f"SELECT COUNT(*) FROM {TARGET_TABLE} WHERE translation_status = 'done' AND (anchor IS NULL OR trim(anchor) = '' OR positive IS NULL OR trim(positive) = '' OR hard_negative IS NULL OR trim(hard_negative) = '')"
).fetchone()[0]
integrity = output_connection.execute('PRAGMA integrity_check').fetchone()[0]
print({
    'source_rows': source_count,
    'translated_rows': done_count,
    'invalid_translations': invalid_count,
    'sqlite_integrity': integrity,
    'output_db': str(OUTPUT_DB),
})
source_connection.close()
output_connection.close()
assert invalid_count == 0
assert integrity == 'ok'